In [8]:
import torch
import torch.nn as nn

class BinaryClassifier(nn.Module):
    def __init__(self, input_size, dropout_rate=0.2, hidden_layers=[256, 128, 64]):
        super().__init__()

        layerlist = []
        n_in = input_size

        for h in hidden_layers:
            layerlist += [
                nn.Linear(n_in, h),
                nn.BatchNorm1d(h),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout_rate),
            ]
            n_in = h

        # Final output layer: 1 logit for binary classification
        layerlist.append(nn.Linear(n_in, 1))

        self.network = nn.Sequential(*layerlist)

        # Better initialization (same as before)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_uniform_(module.weight, nonlinearity='relu')
                nn.init.zeros_(module.bias)

    def forward(self, x):
        # shape: (batch_size, 1) → squeeze to (batch_size,)
        logits = self.network(x).squeeze(-1)
        return logits


In [10]:
import torch
import torch.nn as nn

class BinaryFocalLossWithLogits(nn.Module):
    """
    Binary focal loss operating on logits (recommended).
    - logits: raw model outputs, shape (N,) or (N,1)
    - targets: float tensor in {0,1}, same shape
    Params:
      alpha: None or float in [0,1]. If set, applies class balancing.
             Common: alpha=0.25 if positive class is minority.
      gamma: focusing parameter >= 0. Common: 1.0 or 2.0.
      reduction: 'mean' | 'sum' | 'none'
    """
    def __init__(self, alpha: float | None = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        logits = logits.view(-1)
        targets = targets.view(-1)

        # BCE per-sample (no reduction)
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")

        # p_t = p if y=1 else (1-p)
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)

        # focal modulation
        modulating = (1 - p_t).pow(self.gamma)

        # optional alpha weighting
        if self.alpha is None:
            alpha_factor = 1.0
        else:
            alpha_factor = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        loss = alpha_factor * modulating * bce

        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        elif self.reduction == "none":
            return loss
        else:
            raise ValueError(f"Unknown reduction: {self.reduction}")


In [11]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 0) Load ALREADY-SCALED binary dataset
df_binary = pd.read_csv("artifacts/final_train_scaled_binary_clustered.csv")

TARGET_COL = "MP_binary_high"

# Exclude non-features
exclude = {"SMILES", TARGET_COL, "MW", "MP", "MP_category_default", "Structure_Cluster"}

# Use only numeric feature columns
num_cols = df_binary.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

# Build X/y
X = df_binary[feature_cols].to_numpy(np.float32)
y = df_binary[TARGET_COL].to_numpy(np.float32)

# Stratification labels (cluster)
y_strat = df_binary["Structure_Cluster"].astype(str).to_numpy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("n features:", len(feature_cols))
print("Label counts:", dict(zip(*np.unique(y, return_counts=True))))

# 1) Fix folds once
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr_idx, val_idx) for tr_idx, val_idx in skf.split(X, y_strat)]
print("Built folds:", len(folds))


X shape: (17625, 202)
y shape: (17625,)
n features: 202
Label counts: {np.float32(0.0): np.int64(16853), np.float32(1.0): np.int64(772)}
Built folds: 10


In [12]:
from sklearn.metrics import accuracy_score
from pathlib import Path

#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    

from scipy.special import expit  # stable sigmoid

def save_checkpoint(
    model,
    optimizer,
    epoch,
    train_loss,
    val_loss,
    y_train,
    y_val,
    val_loader,
    fold_idx,
    checkpoint_dir,
    checkpoint_tracking,
    is_final=False,
):
    # Put model in eval mode and collect logits
    model.eval()
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            logits = model(xb).view(-1).cpu().numpy()
            all_logits.append(logits)

    logits_val = np.concatenate(all_logits).reshape(-1)
    probs_val  = expit(logits_val)
    preds_val  = (probs_val >= 0.5).astype(int)

    # y_val expected as numpy 0/1
    y_val_np = np.asarray(y_val).astype(int).reshape(-1)


    # Classification metric: accuracy
    acc = accuracy_score(y_val_np, preds_val)

    # Filename
    if is_final:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
    else:
        checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"

    checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename

    # Save checkpoint (weights + optimizer + losses + metric)
    checkpoint_data = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc,
        "fold_idx": fold_idx,
        "is_final": is_final,
    }
    torch.save(checkpoint_data, checkpoint_path_full)

    # Track info for CSV
    checkpoint_info = {
        "Fold": fold_idx,
        "Epoch": epoch,
        "Checkpoint_Filename": checkpoint_filename,
        "Checkpoint_Path": str(checkpoint_path_full),
        "Train_Loss": round(train_loss, 6),
        "Val_Loss": round(val_loss, 6),
        "Accuracy": round(acc, 6),
        "Is_Final": is_final,
    }
    checkpoint_tracking.append(checkpoint_info)

    checkpoint_type = "Final" if is_final else "Regular"
    print(
        f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - "
        f"Val Loss: {val_loss:.4f} | Acc: {acc:.4f}"
    )
    return True


In [13]:
import copy
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score

def evaluate_fold(
    trial,
    fold_idx,
    X_train_scaled,
    y_train,
    X_val_scaled,
    y_val,
    hidden_layers,
    learning_rate,
    batch_size,
    dropout_rate,
    weight_decay,
    max_epochs=10**9,
    patience=30,
    min_delta=0,
    save_checkpoints=True,
    checkpoint_dir="checkpoints_classifier",
    save_every_n_epochs=15,

    # focal-loss specific (gamma tuned; alpha computed from train fold)
    focal_gamma: float = 2.0,
    alpha_clip: tuple[float, float] = (0.05, 0.95),  # optional safety clamp
):

    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu (binary classifier, FocalLossWithLogits)")

    # ---- checkpoints ----
    checkpoint_tracking = []
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # ---- tensors (y must be float 0/1) ----
    # IMPORTANT: Ensure y_* are 0/1 floats
    y_train = np.asarray(y_train, dtype=np.float32)
    y_val   = np.asarray(y_val, dtype=np.float32)

    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_val_tensor   = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = DataLoader(
        TensorDataset(X_train_tensor, y_train_tensor),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    val_loader = DataLoader(
        TensorDataset(X_val_tensor, y_val_tensor),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    # ---- model ----
    # IMPORTANT: BinaryClassifier must output RAW LOGITS (no sigmoid in the final layer)
    model = BinaryClassifier(
    input_size=X_train_scaled.shape[1],
    hidden_layers=hidden_layers,
    dropout_rate=dropout_rate,).to(device)



    # ==========================================================
    # Compute alpha from TRAIN fold only (counts of high/low)
    # alpha here is the weight applied to y=1 (positive class)
    # Common choice: alpha_pos = N_neg / (N_pos + N_neg)
    # ==========================================================
    with torch.no_grad():
        pos = float(y_train_tensor.sum().item())
        neg = float(y_train_tensor.numel() - y_train_tensor.sum().item())

    # avoid divide-by-zero (shouldn't happen with stratified folds, but safe)
    alpha_pos = neg / (pos + neg + 1e-12)

    # optional clamp (prevents extreme weighting if a fold is weird)
    lo, hi = alpha_clip
    alpha_pos = float(np.clip(alpha_pos, lo, hi))

    print(
        f"[Fold {fold_idx}] pos={pos:.0f} neg={neg:.0f} "
        f"alpha_pos={alpha_pos:.3f} gamma={focal_gamma:.3f}"
    )

    # ---- focal loss ----
    # This replaces BCEWithLogitsLoss(pos_weight=...)
    criterion = BinaryFocalLossWithLogits(alpha=alpha_pos, gamma=focal_gamma, reduction="mean")

    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float("inf")
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

    # ---- training loop ----
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:
            optimizer.zero_grad()

            logits = model(xb).view(-1)  # raw logits
            yb = yb.view(-1)             # targets 0/1 floats

            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # ---- validation ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb).view(-1)
                yb = yb.view(-1)
                loss = criterion(logits, yb)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)

        # track best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        # checkpoints (your existing save_checkpoint() computes sigmoid/acc, so it still works)
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            save_checkpoint(
                model, optimizer, epoch, train_loss, val_loss,
                y_train, y_val, val_loader,
                fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                is_final=False
            )

        # early stopping
        if early_stopper.early_stop(val_loss, epoch=epoch):
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(
                    model, optimizer, epoch, train_loss, val_loss,
                    y_train, y_val, val_loader,
                    fold_idx, fold_checkpoint_dir, checkpoint_tracking,
                    is_final=True
                )
            break

        if epoch % 50 == 0 or epoch == 1:
            print(
                f"[Fold {fold_idx}] Epoch {epoch:4d} | "
                f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
                f"ES {early_stopper.counter}/{patience}"
            )

    # ---- load best ----
    model.load_state_dict(best_state)
    model.eval()

    # save checkpoint tracking csv
    if save_checkpoints and checkpoint_tracking:
        df_ckpt = pd.DataFrame(checkpoint_tracking)
        df_ckpt.to_csv(fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_classifier.csv", index=False)

    # ---- final accuracy on val set (best model) ----
    all_logits = []
    with torch.no_grad():
        for xb, _ in val_loader:
            all_logits.append(model(xb).view(-1).cpu().numpy())

    logits_val = np.concatenate(all_logits)

    # Convert logits -> probabilities for metrics/analysis
    probs_val = 1 / (1 + np.exp(-logits_val))
    preds_val = (probs_val >= 0.5).astype(int)

    y_val_np = np.asarray(y_val).astype(int)
    acc = accuracy_score(y_val_np, preds_val)

    return best_val_loss, acc, model, train_losses, val_losses, stop_epoch


In [14]:
import time
from pathlib import Path

trial_times = []

def objective(trial):
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    # NEW: focal gamma (alpha computed from each training fold)
    focal_gamma = trial.suggest_float("focal_gamma", 1.0, 3.0)

    start = time.perf_counter()

    val_losses = []
    accs = []

    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]

        trial_checkpoint_root = Path("checkpoints_binary_classifier") / f"trial_{trial.number:03d}"

        best_val_loss, acc, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root,

            # NEW:
            focal_gamma=focal_gamma,
        )

        val_losses.append(best_val_loss)
        accs.append(acc)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_val_loss = float(np.mean(val_losses))
    avg_acc      = float(np.mean(accs))
    print(f"Trial {trial.number}: Avg Val Loss = {avg_val_loss:.4f} | Avg Acc = {avg_acc:.4f}")

    return avg_val_loss


In [17]:
import optuna

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
print(best_params)

[I 2026-01-13 22:33:58,994] A new study created in memory with name: no-name-84132e33-5482-4c11-ab6e-c6545f6b5f14


Fold 0: Training on cpu (binary classifier, FocalLossWithLogits)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_000/fold_0
[Fold 0] pos=695 neg=15167 alpha_pos=0.950 gamma=2.123
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.0104 | Acc: 0.7153
[Fold 0] Epoch    1 | Train Loss: 0.0263 | Val Loss: 0.0104 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.0072 | Acc: 0.7504
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.0063 | Acc: 0.8106
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.0059 | Acc: 0.8406
[Fold 0] Epoch   50 | Train Loss: 0.0063 | Val Loss: 0.0060 | ES 5/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.0061 | Acc: 0.8383
[Fold 0] Regular checkpoint saved at epoch 75 - Val Loss: 0.0060 | Acc: 0.8661
[Fold 0] Regular checkpoint saved at epoch 90 - Val Loss: 0.0060 | Acc: 0.8610
[Fold 0] Early stopping at epoch 96 (best Val Loss: 0.0058)
[Fold 0] Final checkpoint saved at epoch 96 - Val L

[I 2026-01-13 22:45:40,856] Trial 0 finished with value: 0.007127268911182073 and parameters: {'dropout_rate': 0.3699372004172302, 'learning_rate': 0.00013492909513209423, 'weight_decay': 3.119424167990617e-05, 'batch_size': 16, 'h1': 224, 'focal_gamma': 2.122572019202612}. Best is trial 0 with value: 0.007127268911182073.


[Fold 9] Regular checkpoint saved at epoch 45 - Val Loss: 0.0076 | Acc: 0.8229
[Fold 9] Early stopping at epoch 45 (best Val Loss: 0.0070)
Trial 0 finished in 11.70 minutes
Trial 0: Avg Val Loss = 0.0071 | Avg Acc = 0.8224
Fold 0: Training on cpu (binary classifier, FocalLossWithLogits)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_001/fold_0
[Fold 0] pos=695 neg=15167 alpha_pos=0.950 gamma=2.125
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.0210 | Acc: 0.7028
[Fold 0] Epoch    1 | Train Loss: 0.0378 | Val Loss: 0.0210 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.0102 | Acc: 0.7533
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.0091 | Acc: 0.7867
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.0084 | Acc: 0.7918
[Fold 0] Epoch   50 | Train Loss: 0.0124 | Val Loss: 0.0083 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.0082 | Acc: 0.7771
[Fold 0] Regular checkpoint saved at epoch 75 

[I 2026-01-13 23:13:02,245] Trial 1 finished with value: 0.007567004074930342 and parameters: {'dropout_rate': 0.3140601670397278, 'learning_rate': 1.1559404335400848e-05, 'weight_decay': 0.00019021153195826978, 'batch_size': 32, 'h1': 192, 'focal_gamma': 2.1252333963748797}. Best is trial 0 with value: 0.007127268911182073.


...

[Fold 9] Early stopping at epoch 95 (best Val Loss: 0.0041)
[Fold 9] Final checkpoint saved at epoch 95 - Val Loss: 0.0043 | Acc: 0.8150
Trial 17 finished in 22.83 minutes
Trial 17: Avg Val Loss = 0.0042 | Avg Acc = 0.8196
Fold 0: Training on cpu (binary classifier, FocalLossWithLogits)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_018/fold_0
[Fold 0] pos=695 neg=15167 alpha_pos=0.950 gamma=2.731
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.0111 | Acc: 0.4594
[Fold 0] Epoch    1 | Train Loss: 0.0237 | Val Loss: 0.0111 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.0059 | Acc: 0.7538
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.0049 | Acc: 0.7879
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.0047 | Acc: 0.7867
[Fold 0] Epoch   50 | Train Loss: 0.0060 | Val Loss: 0.0045 | ES 2/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.0044 | Acc: 0.8361
[Fold 0] Regular checkpoint saved at epoch 75 

[I 2026-01-14 03:22:49,240] Trial 18 finished with value: 0.0049193562115810024 and parameters: {'dropout_rate': 0.25266270849032635, 'learning_rate': 1.9376850399326684e-05, 'weight_decay': 0.00085488489512568, 'batch_size': 16, 'h1': 224, 'focal_gamma': 2.7312373708419377}. Best is trial 17 with value: 0.004188681257657154.


[Fold 9] Early stopping at epoch 70 (best Val Loss: 0.0049)
[Fold 9] Final checkpoint saved at epoch 70 - Val Loss: 0.0051 | Acc: 0.7832
Trial 18 finished in 25.51 minutes
Trial 18: Avg Val Loss = 0.0049 | Avg Acc = 0.8229
Fold 0: Training on cpu (binary classifier, FocalLossWithLogits)
Checkpoints will be saved to: checkpoints_binary_classifier/trial_019/fold_0
[Fold 0] pos=695 neg=15167 alpha_pos=0.950 gamma=1.854
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.0263 | Acc: 0.1225
[Fold 0] Epoch    1 | Train Loss: 0.0431 | Val Loss: 0.0263 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.0123 | Acc: 0.8111
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.0114 | Acc: 0.8917
[Fold 0] Regular checkpoint saved at epoch 45 - Val Loss: 0.0100 | Acc: 0.8667
[Fold 0] Epoch   50 | Train Loss: 0.0131 | Val Loss: 0.0103 | ES 1/30
[Fold 0] Regular checkpoint saved at epoch 60 - Val Loss: 0.0097 | Acc: 0.8446
[Fold 0] Regular checkpoint saved at epoch 75 

[I 2026-01-14 03:49:36,538] Trial 19 finished with value: 0.009355800278481674 and parameters: {'dropout_rate': 0.31504511372125354, 'learning_rate': 1.8905251930428023e-05, 'weight_decay': 8.69832837377282e-06, 'batch_size': 16, 'h1': 64, 'focal_gamma': 1.8543973478521396}. Best is trial 17 with value: 0.004188681257657154.


[Fold 9] Early stopping at epoch 188 (best Val Loss: 0.0097)
[Fold 9] Final checkpoint saved at epoch 188 - Val Loss: 0.0102 | Acc: 0.7582
Trial 19 finished in 26.79 minutes
Trial 19: Avg Val Loss = 0.0094 | Avg Acc = 0.8025
{'dropout_rate': 0.24406688691376077, 'learning_rate': 1.8766660477174088e-05, 'weight_decay': 7.198869254318667e-06, 'batch_size': 16, 'h1': 160, 'focal_gamma': 2.9906584574948836}


In [18]:
print("Best Trial Number:", study.best_trial.number)
print(best_params)

Best Trial Number: 17
{'dropout_rate': 0.24406688691376077, 'learning_rate': 1.8766660477174088e-05, 'weight_decay': 7.198869254318667e-06, 'batch_size': 16, 'h1': 160, 'focal_gamma': 2.9906584574948836}


In [19]:
from pathlib import Path
import torch
import pandas as pd

BASE = Path.cwd()
artifacts_dir = BASE / "artifacts"

best_models_dir = artifacts_dir / "binary_focal_loss_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_binary_focal_loss_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

print("Best hyperparameters from Optuna:", best_params)

def build_hidden_layers_from_best(best_params):
    h1 = best_params["h1"]
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    return [h1, h2, h3]

hidden_layers = build_hidden_layers_from_best(best_params)
dropout_rate  = best_params["dropout_rate"]
learning_rate = best_params["learning_rate"]
weight_decay  = best_params["weight_decay"]
batch_size    = best_params["batch_size"]

# NEW: focal gamma from Optuna
focal_gamma   = best_params["focal_gamma"]

print("Using hidden_layers:", hidden_layers)
print("dropout:", dropout_rate, "| lr:", learning_rate, "| wd:", weight_decay,
      "| batch_size:", batch_size, "| focal_gamma:", focal_gamma)

final_fold_metrics = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n==================== Final training for fold {fold_idx} ====================")

    X_train_scaled = X[tr_idx]
    y_train        = y[tr_idx]
    X_val_scaled   = X[val_idx]
    y_val          = y[val_idx]

    best_val_loss, acc, model, train_losses, val_losses, stop_epoch = evaluate_fold(
        trial=None,
        fold_idx=fold_idx,
        X_train_scaled=X_train_scaled,
        y_train=y_train,
        X_val_scaled=X_val_scaled,
        y_val=y_val,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        max_epochs=10**9,
        patience=30,
        min_delta=0.0,
        save_checkpoints=True,
        checkpoint_dir=final_ckpt_dir,
        save_every_n_epochs=15,

        # NEW:
        focal_gamma=focal_gamma,
    )

    model_path = best_models_dir / f"binary_focal_loss_best_fold_{fold_idx}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "focal_gamma": float(focal_gamma),
            "fold_idx": fold_idx,
            "best_val_focal_loss": float(best_val_loss),
            "val_accuracy": float(acc),
        },
        model_path,
    )

    print(f"[Fold {fold_idx}] Saved best classifier model to: {model_path}")
    print(f"[Fold {fold_idx}] Val FocalLoss: {best_val_loss:.4f} | Val Acc: {acc:.4f} | Stop epoch: {stop_epoch}")

    final_fold_metrics.append(
        {
            "Fold": fold_idx,
            "Best_Val_FocalLoss": float(best_val_loss),
            "Val_Accuracy": float(acc),
            "Stop_Epoch": stop_epoch,
            "Model_Path": str(model_path),
        }
    )

metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "classifier_focal_loss_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)

print("\n✅ Saved summary of best classifier focal loss models across folds to:", metrics_path)
print(metrics_df)

Best hyperparameters from Optuna: {'dropout_rate': 0.24406688691376077, 'learning_rate': 1.8766660477174088e-05, 'weight_decay': 7.198869254318667e-06, 'batch_size': 16, 'h1': 160, 'focal_gamma': 2.9906584574948836}
Using hidden_layers: [160, 80, 40]
dropout: 0.24406688691376077 | lr: 1.8766660477174088e-05 | wd: 7.198869254318667e-06 | batch_size: 16 | focal_gamma: 2.9906584574948836

==================== Final training for fold 0 ====================
Fold 0: Training on cpu (binary classifier, FocalLossWithLogits)
Checkpoints will be saved to: /Users/sdl5_mp/Documents/SDL5_MP/transfer_learning/checkpoints_binary_focal_loss_best/fold_0
[Fold 0] pos=695 neg=15167 alpha_pos=0.950 gamma=2.991
[Fold 0] Regular checkpoint saved at epoch 1 - Val Loss: 0.0106 | Acc: 0.4317
[Fold 0] Epoch    1 | Train Loss: 0.0271 | Val Loss: 0.0106 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - Val Loss: 0.0046 | Acc: 0.8338
[Fold 0] Regular checkpoint saved at epoch 30 - Val Loss: 0.0044 | Acc: 0

In [20]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit

confusion_matrices = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== Confusion matrix for fold {fold_idx} =====")

    X_val = X[val_idx]
    y_val = y[val_idx].astype(int).reshape(-1)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

    # FIXED: correct focal-loss model filename
    model_path = best_models_dir / f"binary_focal_loss_best_fold_{fold_idx}.pt"
    checkpoint = torch.load(model_path, map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=checkpoint["hidden_layers"],
        dropout_rate=checkpoint["dropout_rate"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    with torch.no_grad():
        logits = model(X_val_tensor).view(-1).cpu().numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    confusion_matrices.append(cm)

    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)

    tn, fp, fn, tp = cm.ravel()
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")



===== Confusion matrix for fold 0 =====
Confusion matrix (rows=true, cols=pred):
[[1409  277]
 [   8   69]]
TN=1409, FP=277, FN=8, TP=69

===== Confusion matrix for fold 1 =====
Confusion matrix (rows=true, cols=pred):
[[1376  317]
 [  10   60]]
TN=1376, FP=317, FN=10, TP=60

===== Confusion matrix for fold 2 =====
Confusion matrix (rows=true, cols=pred):
[[1392  293]
 [   8   70]]
TN=1392, FP=293, FN=8, TP=70

===== Confusion matrix for fold 3 =====
Confusion matrix (rows=true, cols=pred):
[[1391  310]
 [  10   52]]
TN=1391, FP=310, FN=10, TP=52

===== Confusion matrix for fold 4 =====
Confusion matrix (rows=true, cols=pred):
[[1332  342]
 [  11   78]]
TN=1332, FP=342, FN=11, TP=78

===== Confusion matrix for fold 5 =====
Confusion matrix (rows=true, cols=pred):
[[1411  276]
 [   5   70]]
TN=1411, FP=276, FN=5, TP=70

===== Confusion matrix for fold 6 =====
Confusion matrix (rows=true, cols=pred):
[[1317  344]
 [   9   92]]
TN=1317, FP=344, FN=9, TP=92

===== Confusion matrix for fol

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_43517/2956563340.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_loca

NameError: name 'all_fold_metrics' is not defined

In [23]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
)

In [24]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import confusion_matrix
from scipy.special import expit

# store per-fold metrics
metrics_per_fold = []

for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n===== Fold {fold_idx} =====")

    # ---- validation data ----
    X_val = X[val_idx]
    y_val = y[val_idx].astype(int).reshape(-1)

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)

    # ---- load model ----
    model_path = best_models_dir / f"binary_focal_loss_best_fold_{fold_idx}.pt"
    checkpoint = torch.load(model_path, map_location="cpu")

    model = BinaryClassifier(
        input_size=X.shape[1],
        hidden_layers=checkpoint["hidden_layers"],
        dropout_rate=checkpoint["dropout_rate"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    # ---- inference ----
    with torch.no_grad():
        logits = model(X_val_tensor).view(-1).cpu().numpy()

    probs = expit(logits)
    preds = (probs >= 0.5).astype(int)

    # ---- confusion matrix ----
    cm = confusion_matrix(y_val, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    print("Confusion matrix (rows=true, cols=pred):")
    print(cm)
    print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    # ---- metrics ----
    acc = accuracy_score(y_val, preds)
    bal_acc = balanced_accuracy_score(y_val, preds)
    precision = precision_score(y_val, preds, zero_division=0)
    recall = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)

    # ROC-AUC uses probabilities (only valid if both classes present)
    if len(np.unique(y_val)) == 2:
        auc = roc_auc_score(y_val, probs)
    else:
        auc = np.nan

    print(f"Accuracy:          {acc:.3f}")
    print(f"Balanced accuracy: {bal_acc:.3f}")
    print(f"Precision (high):  {precision:.3f}")
    print(f"Recall (high):     {recall:.3f}")
    print(f"F1 score:          {f1:.3f}")
    print(f"ROC-AUC:           {auc:.3f}")

    metrics_per_fold.append({
        "fold": fold_idx,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": auc,
    })



===== Fold 0 =====
Confusion matrix (rows=true, cols=pred):
[[1409  277]
 [   8   69]]
TN=1409, FP=277, FN=8, TP=69
Accuracy:          0.838
Balanced accuracy: 0.866
Precision (high):  0.199
Recall (high):     0.896
F1 score:          0.326
ROC-AUC:           0.944

===== Fold 1 =====
Confusion matrix (rows=true, cols=pred):
[[1376  317]
 [  10   60]]
TN=1376, FP=317, FN=10, TP=60
Accuracy:          0.815
Balanced accuracy: 0.835
Precision (high):  0.159
Recall (high):     0.857
F1 score:          0.268
ROC-AUC:           0.909

===== Fold 2 =====
Confusion matrix (rows=true, cols=pred):
[[1392  293]
 [   8   70]]
TN=1392, FP=293, FN=8, TP=70
Accuracy:          0.829
Balanced accuracy: 0.862
Precision (high):  0.193
Recall (high):     0.897
F1 score:          0.317
ROC-AUC:           0.931

===== Fold 3 =====
Confusion matrix (rows=true, cols=pred):
[[1391  310]
 [  10   52]]
TN=1391, FP=310, FN=10, TP=52
Accuracy:          0.818
Balanced accuracy: 0.828
Precision (high):  0.144
Recal

/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_43517/561640148.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_locat

Confusion matrix (rows=true, cols=pred):
[[1399  290]
 [  10   63]]
TN=1399, FP=290, FN=10, TP=63
Accuracy:          0.830
Balanced accuracy: 0.846
Precision (high):  0.178
Recall (high):     0.863
F1 score:          0.296
ROC-AUC:           0.931

===== Fold 8 =====
Confusion matrix (rows=true, cols=pred):
[[1430  251]
 [  11   70]]
TN=1430, FP=251, FN=11, TP=70
Accuracy:          0.851
Balanced accuracy: 0.857
Precision (high):  0.218
Recall (high):     0.864
F1 score:          0.348
ROC-AUC:           0.945

===== Fold 9 =====
Confusion matrix (rows=true, cols=pred):
[[1429  267]
 [   9   57]]
TN=1429, FP=267, FN=9, TP=57
Accuracy:          0.843
Balanced accuracy: 0.853
Precision (high):  0.176
Recall (high):     0.864
F1 score:          0.292
ROC-AUC:           0.918


/var/folders/4j/q1b8spls57ldz2j3g8k6nxh40000gn/T/ipykernel_43517/561640148.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_locat